# ДЗ 2 — Птичья платформа

In [213]:
import pymc as pm
import arviz as az
import numpy as np
import seaborn as sns
import xarray as xa
import matplotlib.pyplot as plt
import json

Вы управляете некой «птичьей» платформой и экспериментируете с рекомендательной системой и системой цензуры.

Улучшение рекомендаций или цензуры приводит к

* росту/снижению вовлечённости в рекомендации (**люди переходят в тред**)
* росту/снижению **времени чтения** (люди дольше читают треды)
* Есть **кнопки лайков**, нажимая которые, люди вовлекаются ещё больше

Немного исторической справки

* Историческая вовлечённость в рекомендации — **5%**
* Историческое время чтения распределено **экспоненциально**, историческое среднее время чтения — 1 мин (60 с)
* **40%** стартовых постов тредов получают лайк

```
показ -> 5% -> читает тред ~ Exponential(1/60) -> 40% -> лайк
      \-> 95% -> листает дальше                \-> 60% без лайка
```

Когда-то давно вы обсуждали возможные интервалы исходов, и они такие:

* **+/- 3.5%** — изменение доли прочтений треда для всех экспериментов, **кроме D**. Для D вы рассматривали **+/- 5%**, потому что он затрагивал чувствительную тему цензуры
* **+/- 3%** — для времени чтения, одинаково для всех экспериментов
* **+/- 1.5%** — для всех метрик, связанных с лайками.

**Все изменения ОТНОСИТЕЛЬНЫЕ**

* относительное изменение на **+/- 3%** считается достаточно значимым

Оценивание (всего 40 баллов)
1. Модель: 8 баллов
    1. Реализуйте модель (2 балла)
    2. Отобразите структуру модели (2 балла)
    3. Сэмплируйте из неё, дивергенций быть не должно (2 балла)
    4. Постройте az.forest_plot с ROPE (2 балла)
2. Принятие решений:
    * 8 задач, по 4 балла каждая
    * Каждый график нужно прокомментировать
3. Бонусных баллов в этот раз нет

In [ ]:
data = json.load(open("ha-2-data.json", "r"))

**Какой эксперимент лучший?**

1. Попробуйте ответить, глядя только на график с ROPE...

2. Эксперимент, который максимизирует число лайков *после рекомендации*

3. Эксперимент, который максимизирует время чтения *после рекомендации*

4. Учтите денежную оценку (всё одновременно)
    1. каждый переход в тред — \$0.005
    2. каждая секунда чтения — \$0.0005
    3. каждый лайк — \$0.0015
    4. число пользователей N = 100000000

5. Учтите затраты на внедрение проектов вместе с предыдущей информацией
    1. 3000
    2. 2000
    3. 1500
    4. 1000

6. Вам предстоит столкнуться с тем, что люди переходят на другую платформу. Будущая вовлечённость в целом довольно неопределённа. Вот сценарии, которые вы рассматриваете:
    1. Оптимистичный: впереди светлое будущее, люди будут заходить в приложение на 5\% чаще
    2. Консервативный: ничего не изменится
    3. Пессимистичный: лидеры мнений уйдут на новую платформу и заберут 5\% аудитории, затраты на внедрение вырастут на 50%
    4. Депрессивный: лидеры мнений уйдут на новую платформу и заберут 15\% аудитории, затраты на внедрение вырастут в 7 раз

7. Для каждого сценария найдите вероятность того, что эксперимент окажется лучшим среди всех экспериментов. Покажите тепловую карту (heatmap); **Следите за подписями осей x/y, убедитесь, что они не перепутаны**


8. Решение нужно принимать консервативно: найдите нижний 5%-квантиль для каждого эксперимента и сценария, покажите его на тепловой карте с центром в нуле. **Следите за подписями осей x/y, убедитесь, что они не перепутаны**